In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn import preprocessing, model_selection, feature_selection
from sklearn.decomposition import PCA
import mytrain_lib_cluster as ml
import goalnets as gn
import os

In [2]:
import importlib
importlib.reload(ml)
importlib.reload(gn)

<module 'goalnets' from 'f:\\TFG\\code\\goalnets.py'>

In [7]:
import importlib
importlib.reload(ml)

dataset = 'historical_longterm'
drop = []
regression = False

config = {
    'root':'test_init_true',
    'train_data':ml.FootballMatchesDataset('train',dataset,drop=drop,factor=-1, regression=regression),
    'test_data':ml.FootballMatchesDataset('test',dataset,drop=drop,factor=-1, regression=regression),
    # 'train_data':ml.WyscoutDataset("train"),
    # 'test_data':ml.WyscoutDataset("test"),
    'model':{
        'model': gn.NeuralNetworkDOClass,
        'params':{
            'hidden_neurons':[20,4,3],
            'activation': F.relu,
            'output_classes': 2 if regression else 3,
            'activ_opt':{
                'negative_slope':0.
            },
            'loss_weights':torch.tensor([1.1,0.9,1.0]),
            'initialization':True,
            'p':0.071718596
        }
    },
    'optimizer':{
        'optimizer':torch.optim.Adam,
        'params':{
            # 'momentum':0,
            # 'weight_decay':0.2,
            # 'nesterov':0,
            # 'dampening':0.284022281,
            'lr':0.1,
            'betas':(0.397,0.99)
            # 'b2':
        }
    },
    'criterion':nn.CrossEntropyLoss,
    'scalers':preprocessing.Normalizer,
    'batch_size':32,
    'epochs':10,
    'save_outputs':False,
    'display':True,
    'logs':True,
}

In [8]:
import importlib
importlib.reload(ml)
importlib.reload(gn)
ml.Tuning(**config)

Config: 1/1: loss train: 1.06, accuracy train: 0.46, loss test: 1.06, accuracy test: 0.46		t: 0.46


1

In [7]:
model = gn.dumbmodelclass(config['train_data'],[20,10,5],3,loss_func=nn.CrossEntropyLoss)
params = sum([x.nelement() for x in model.model.parameters()])
print(f"The model has {params} parameters.",end='\n')


The model has 3103 parameters.


##### BN

In [44]:
# MULTIPLE EXECS
config['root']  = "rs_wysc_basic_lwd2"
config['model']['params']['loss_weights'] = torch.tensor([.9,.2,.3])
ml.Tuning(**config)
config['root']  = "rs_wysc_basic_lwh2"
config['model']['params']['loss_weights'] = torch.tensor([.2,.9,.3])
ml.Tuning(**config)

Config: 1/1: loss train: 0.38, accuracy train: 0.32, loss test: 0.41, accuracy test: 0.27		: 0.27
Config: 1/1: loss train: 0.36, accuracy train: 0.49, loss test: 0.42, accuracy test: 0.46		: 0.4645


1

##### WD

In [14]:
config['model']['params']['hidden_neurons'] = [50,30,10]
config['model']['model'] = gn.dumbmodelclass

# config['root']  = "wysc_3000_dumb"
# config['optimizer']['params']['weight_decay'] = 0
# ml.Tuning(**config)
config['root']  = "wysc_3000_dumb_wd0.1"
config['optimizer']['params']['weight_decay'] = 0.1
ml.Tuning(**config)
config['root']  = "wysc_3000_dumb_wd0.5"
config['optimizer']['params']['weight_decay'] = 0.5
ml.Tuning(**config)
config['root']  = "wysc_3000_dumb_wd2.0"
config['optimizer']['params']['weight_decay'] = 2.0
ml.Tuning(**config)

Config: 1/1: loss train: 1.07, accuracy train: 0.46, loss test: 1.07, accuracy test: 0.45		: 0.45
Config: 1/1: loss train: 1.08, accuracy train: 0.46, loss test: 1.08, accuracy test: 0.45		: 0.4530
Config: 1/1: loss train: 1.09, accuracy train: 0.46, loss test: 1.09, accuracy test: 0.45		: 0.4530


1

##### FACTORES

In [4]:
# config['root']  = "hl_fno"
# config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=-1, regression=regression)
# config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=-1, regression=regression)
# ml.Tuning(**config)

# config['root']  = "hl_f0"
# config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=0, regression=regression)
# config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=0, regression=regression)
# ml.Tuning(**config)

config['root']  = "hl_f1"
config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=10, regression=regression)
config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=10, regression=regression)
ml.Tuning(**config)

config['root']  = "hl_f2"
config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=20, regression=regression)
config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=20, regression=regression)
ml.Tuning(**config)

config['root']  = "hl_f5"
config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=50, regression=regression)
config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=50, regression=regression)
ml.Tuning(**config)

config['root']  = "hl_f10"
config['train_data'] = ml.FootballMatchesDataset('train',dataset,drop=drop,factor=100, regression=regression)
config['test_data']  =  ml.FootballMatchesDataset('test',dataset,drop=drop,factor=100, regression=regression)
ml.Tuning(**config)

Config: 1/1: loss train: 0.14, accuracy train: 0.48, loss test: 0.28, accuracy test: 0.50		t: 0.50
Config: 1/1: loss train: 0.28, accuracy train: 0.48, loss test: 0.56, accuracy test: 0.50		t: 0.50.26
Config: 1/1: loss train: 0.71, accuracy train: 0.48, loss test: 1.40, accuracy test: 0.50		t: 0.50.26
Config: 1/1: loss train: 1.42, accuracy train: 0.48, loss test: 2.79, accuracy test: 0.50		t: 0.50.26


1

##### Best accuracy

In [4]:
import importlib
importlib.reload(ml)

dataset = 'historical_longterm'
drop = []
regression = False

config = {
    'root':'wysc_best',
    'train_data':ml.FootballMatchesDataset('train',dataset,drop=drop,factor=-1, regression=regression),
    'test_data':ml.FootballMatchesDataset('test',dataset,drop=drop,factor=-1, regression=regression),
    # 'train_data':ml.WyscoutDataset("train"),
    # 'test_data':ml.WyscoutDataset("test"),
    'model':{
        'model': gn.NeuralNetworkDOClass,
        'params':{
            'hidden_neurons':[51,6,7],
            'activation': F.selu,
            'output_classes': 2 if regression else 3,
            'activ_opt':{
                'negative_slope':0.
            },
            'loss_weights':torch.tensor([1.1,0.9,1.0]),
            'initialization':False,
            'p':0.136341944633018
        }
    },
    'optimizer':{
        'optimizer':torch.optim.SGD,
        'params':{
            'momentum':0,
            'weight_decay':0,
            'nesterov':0,
            'dampening':0.161764655710075,
            'lr':0.0001,
            # 'betas':(0.397,0.99)
            # 'b2':
        }
    },
    'criterion':nn.CrossEntropyLoss,
    'scalers':preprocessing.StandardScaler,
    'batch_size':32,
    'epochs':75,
    'save_outputs':False,
    'display':True,
    'logs':True,
}

In [5]:
config['root']  = "top_lr1"
config['optimizer']['params']['lr'] = 0.0001
ml.Tuning(**config)

config['root']  = "top_lr2"
config['optimizer']['params']['lr'] = 0.01
ml.Tuning(**config)

config['root']  = "top_lr3"
config['optimizer']['params']['lr'] = .1
ml.Tuning(**config)

config['root']  = "top_lr4"
config['optimizer']['params']['lr'] = 1.
ml.Tuning(**config)

Config: 1/1: loss train: 1.00, accuracy train: 0.52, loss test: 0.99, accuracy test: 0.51		t: 0.51


RuntimeError: CUDA out of memory. Tried to allocate 492.00 MiB (GPU 0; 4.00 GiB total capacity; 1.64 GiB already allocated; 0 bytes free; 3.26 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF